# Performance Optimization in NumPy

Writing efficient NumPy code is critical for data science and ML workloads.

**Topics:** vectorization, `out` parameter, `einsum`, ufuncs (reduce/accumulate), memory layout, `np.vectorize` caveats, profiling.

## Vectorization is Key

Always prefer vectorized operations over explicit Python loops.

In [ ]:
import numpy as np

n = 10_000_000
arr = np.arange(n, dtype=np.float64)

# Loop vs vectorized comparison
import time

# Vectorized
start = time.time()
result_vec = np.sin(arr) + np.cos(arr) * 2.0
vec_time = time.time() - start
print(f"Vectorized: {vec_time:.4f}s")

# For comparison, even a simple loop is much slower
small = np.arange(100_000, dtype=np.float64)
start = time.time()
result_loop = np.empty_like(small)
for i in range(len(small)):
    result_loop[i] = np.sin(small[i]) + np.cos(small[i]) * 2.0
loop_time = time.time() - start
print(f"Loop (100k): {loop_time:.4f}s (extrapolated ~{loop_time*100:.1f}s for 10M)")

## The `out` Parameter (In-Place Operations)

Avoid allocating new memory by writing results to an existing array.

In [ ]:
n = 5_000_000
a = np.random.rand(n)
b = np.random.rand(n)
out = np.empty(n)

# Without out: creates new array
start = time.time()
c = a + b
t1 = time.time() - start

# With out: writes to existing array
start = time.time()
np.add(a, b, out=out)
t2 = time.time() - start

print(f"Without out: {t1*1000:.2f}ms")
print(f"With out:    {t2*1000:.2f}ms")
print(f"Results match: {np.allclose(c, out)}")

# In-place operations
x = np.arange(5, dtype=float)
x += 10  # modifies x in-place
print("In-place:", x)

## Einstein Summation (einsum)

`einsum` is a compact, expressive way to write many linear algebra operations. It can be faster than multiple functions.

In [ ]:
A = np.random.rand(3, 4)
B = np.random.rand(4, 5)

# Matrix multiplication
C_dot = A @ B
C_einsum = np.einsum('ij,jk->ik', A, B)
print("Matrix mult match:", np.allclose(C_dot, C_einsum))

# Trace of matrix product
trace = np.einsum('ij,ji->', A, B[:3, :4])  # note: shapes trimmed
print(f"Trace via einsum: {trace:.4f}")

# Outer product
v = np.array([1, 2, 3])
w = np.array([4, 5, 6])
outer = np.einsum('i,j->ij', v, w)
print("\nOuter via einsum:\n", outer)

# Batch matrix multiplication
batches = np.random.rand(10, 3, 4)
B = np.random.rand(4, 5)
result = np.einsum('bij,jk->bik', batches, B)
print(f"\nBatch matmul shape: {result.shape}")

In [ ]:
# Performance: einsum can be faster than chaining ops
a = np.random.rand(200, 300)
b = np.random.rand(300, 200)
c = np.random.rand(200, 200)

# Chained: (a @ b) * c
start = time.time()
r1 = (a @ b) * c
t_chain = time.time() - start

# einsum: combine into single operation
start = time.time()
r2 = np.einsum('ij,jk,ik->ik', a, b, c, optimize=True)
t_einsum = time.time() - start

print(f"Chained: {t_chain*1000:.2f}ms")
print(f"einsum:  {t_einsum*1000:.2f}ms")
print(f"Match: {np.allclose(r1, r2)}")

## Universal Functions (ufuncs) — reduce, accumulate

In [ ]:
# ufunc.reduce
arr = np.array([1, 2, 3, 4, 5])
print("Sum via add.reduce:", np.add.reduce(arr))        # 15
print("Product via multiply.reduce:", np.multiply.reduce(arr))  # 120
print("Logical and via logical_and.reduce:", np.logical_and.reduce([True, True, False]))

# ufunc.accumulate (running total)
print("\nCumulative sum:", np.add.accumulate(arr))

# ufunc.outer
print("\nOuter sum:")
print(np.add.outer(np.array([1, 2]), np.array([10, 20, 30])))

# ufunc.at (unbuffered in-place op at specific indices)
x = np.array([0, 0, 0])
np.add.at(x, [0, 1, 0, 2, 0], 1)
print("\nadd.at result:", x)  # [3, 1, 1]

## Memory Layout: C-contiguous vs Fortran-contiguous

Memory layout affects cache performance. Default is C-contiguous (row-major).

In [ ]:
arr = np.arange(1000000).reshape(1000, 1000)
arr_f = np.asfortranarray(arr)  # or arr.T, or order='F'

print("C-contiguous:", arr.flags['C_CONTIGUOUS'])
print("F-contiguous:", arr.flags['F_CONTIGUOUS'])
print("\nF-contiguous copy:", arr_f.flags['F_CONTIGUOUS'])

# Iterate by rows (C order = efficient)
start = time.time()
row_sum_c = np.sum(arr, axis=1)
t_c = time.time() - start

# Iterate by columns on C array (inefficient - non-contiguous access)
start = time.time()
col_sum_c = np.sum(arr, axis=0)
t_c_col = time.time() - start

# Col sums on F array (efficient for columns)
start = time.time()
col_sum_f = np.sum(arr_f, axis=0)
t_f_col = time.time() - start

print(f"\nC-contiguous row sums: {t_c*1000:.2f}ms")
print(f"C-contiguous col sums: {t_c_col*1000:.2f}ms")
print(f"F-contiguous col sums: {t_f_col*1000:.2f}ms")

## np.vectorize — Not a Performance Tool

`np.vectorize` is a convenience wrapper around loops — it is **not** true vectorization.

In [ ]:
def f(x):
    return x**2 + 2*x + 1

arr = np.arange(1_000_000)

# True vectorization
start = time.time()
r1 = f(arr)
t_vec = time.time() - start

# np.vectorize (still loops in C at decorator level, but slower than vectorized)
f_vec = np.vectorize(f)
start = time.time()
r2 = f_vec(arr)
t_vec_wrap = time.time() - start

# Manual loop
start = time.time()
r3 = np.empty_like(arr)
for i in range(len(arr)):
    r3[i] = f(arr[i])
t_loop = time.time() - start

print(f"True vectorized: {t_vec*1000:.1f}ms")
print(f"np.vectorize:    {t_vec_wrap*1000:.1f}ms")
print(f"Python loop:     {t_loop*1000:.1f}ms")

## Profiling with %timeit (Magic Command)

In Jupyter, `%timeit` and `%%timeit` give accurate timing via multiple runs.

In [ ]:
# %timeit line magic (run this cell in a notebook)
# %timeit np.sum(np.random.rand(100000))

# %%timeit cell magic
# %%timeit
# a = np.random.rand(1000, 1000)
# b = np.random.rand(1000, 1000)
# c = a @ b

# Manual timing for this environment
import timeit
n = 100
stmt = "np.sum(np.random.rand(10000))"
timer = timeit.timeit(stmt, globals=globals(), number=n)
print(f"Average: {timer/n*1e6:.1f} µs per call")

# Compare methods
setup = "import numpy as np; arr=np.random.rand(10000)"
t_sum = timeit.timeit("np.sum(arr)", setup=setup, number=10000)
t_mean = timeit.timeit("np.mean(arr)", setup=setup, number=10000)
print(f"\nnp.sum:  {t_sum/10:.3f}ms per 10k calls")
print(f"np.mean: {t_mean/10:.3f}ms per 10k calls")

## Problem 1: Optimize Matrix Operations

Compute `trace(A @ B @ C)` efficiently.

In [ ]:
A = np.random.rand(500, 300)
B = np.random.rand(300, 400)
C = np.random.rand(400, 500)

# Naive: compute full product then trace
start = time.time()
r1 = np.trace(A @ B @ C)
t_naive = time.time() - start

# Efficient: einsum avoids forming large intermediate matrices
start = time.time()
r2 = np.einsum('ij,jk,ki->', A, B, C, optimize=True)
t_einsum = time.time() - start

# Alternative: trace(AB @ C) = trace(C @ AB) = sum of element-wise
start = time.time()
r3 = np.sum((A @ B) * C.T)
t_alt = time.time() - start

print(f"Naive:   {t_naive*1000:.1f}ms  result={r1:.4f}")
print(f"Einsum:  {t_einsum*1000:.1f}ms  result={r2:.4f}")
print(f"Alt:     {t_alt*1000:.1f}ms  result={r3:.4f}")
print(f"All match: {np.allclose([r1, r2, r3], r1)}")

## Problem 2: Pairwise Euclidean Distance Matrix

Compare loop vs broadcasting for computing all pairwise distances.

In [ ]:
n_points = 2000
n_dims = 100
points = np.random.rand(n_points, n_dims)

# Broadcasting approach
start = time.time()
diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
dist_broadcast = np.sqrt(np.sum(diff**2, axis=-1))
t_bc = time.time() - start

# Using einsum (|x-y|^2 = |x|^2 + |y|^2 - 2*x*y)
start = time.time()
x_norm = np.sum(points**2, axis=1)[:, np.newaxis]
y_norm = np.sum(points**2, axis=1)[np.newaxis, :]
xy = points @ points.T
dist_einsum = np.sqrt(np.maximum(0, x_norm + y_norm - 2 * xy))  # clamp to avoid negative due to rounding
t_fast = time.time() - start

print(f"Broadcasting: {t_bc*1000:.1f}ms")
print(f"Fast formula: {t_fast*1000:.1f}ms")
print(f"Speedup: {t_bc/t_fast:.1f}x")
print(f"Match: {np.allclose(dist_broadcast, dist_einsum, atol=1e-10)}")

## Summary

1. **Vectorize everything** — avoid Python loops
2. **Use `out` parameter** — avoid unnecessary allocations
3. **`einsum`** — compact, expressive, can be faster than chained ops
4. **Ufunc methods** (`reduce`, `accumulate`, `outer`, `at`) — powerful building blocks
5. **Memory layout matters** — iterate in memory order for cache efficiency
6. **`np.vectorize` is NOT fast** — use true vectorized operations
7. **Profile** with `%timeit` or `timeit` module to make data-driven decisions